# **Project 30:** *Analyzis of Wordnet consistency in Synonymy and Antonymy relationships*
**Authors:**
* Miro Lammi (2208316), mlammi23@student.oulu.fi
* Eemeli Kyröläinen (2309218), ekyrolai23@student.oulu.fi

#### Requirements (recommended *Python 3.12*):
`pip install nltk` <br>
`pip install pandas` <br>
`pip install jupyter` <br>
`pip install matplotlib` <br>
`pip install numpy` <br>
`pip install gensim` <br>
`pip install scikit-learn` <br>
`pip install requests` <br>
`pip install scipy` <br>


or run: <br>
`pip install -r requirements.txt`

## Imports and initialization

In [ ]:
import requests
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gensim.downloader as api

from collections import defaultdict
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import ttest_ind

from nltk.corpus import wordnet as wn

POS_TAGS = [wn.NOUN, wn.VERB]
RELATIONS = {"synonym", "antonym"}
METHODS = ["path", "wup"]
COLORS = {
    ("synonym", wn.NOUN): "#4682B4",
    ("synonym", wn.VERB): "#8DB9D8",
    ("antonym", wn.NOUN): "#DC143C",
    ("antonym", wn.VERB): "#F08080",
}
URL = "https://www.gutenberg.org/cache/epub/51155/pg51155.txt"


def similarity(s1, s2, method="path"):
    if method == "path":
        return wn.synset(s1).path_similarity(wn.synset(s2))
    elif method == "wup":
        return wn.synset(s1).wup_similarity(wn.synset(s2))
    else:
        raise ValueError("[ERROR]: Invalid method! method must be one of {}".format(METHODS))

## 1. Getting synonyms and antonyms for each verb and noun in WordNet lexical database and calculating path similarity for each pair.

In [ ]:
def get_all_pairs(pos):
    pairs = set()

    for synset in wn.all_synsets(pos):
        lemmas = synset.lemmas()

        for l1, l2 in combinations(lemmas, 2):
            w1, w2 = sorted([l1.name(), l2.name()])

            if w1 != w2:
                pairs.add((pos, "synonym", w1, w2, synset.name(), synset.name()))

        for lemma in lemmas:
            for antonym in lemma.antonyms():
                a_synset = antonym.synset()
                if a_synset.pos() != pos:
                    continue

                a, b = sorted([(lemma.name(), synset.name()),
                               (antonym.name(), a_synset.name())])

                if a[0] != b[0]:
                    pairs.add((pos, "antonym", a[0], b[0], a[1], b[1]))
    return pairs

results = []

for pos in POS_TAGS:
    pairs = get_all_pairs(pos)

    for p, rel, w1, w2, s1, s2 in pairs:
        sim = similarity(s1, s2, "wup")
        results.append({
            'pos': p,
            'relation': rel,
            'word1': w1,
            'word2': w2,
            's1': s1,
            's2': s2,
            'similarity': sim
        })

DATAFRAME = pd.DataFrame(results)

statistics = (DATAFRAME.groupby(["pos", "relation"])["similarity"]
              .agg(["count", "min", "max", "mean"]).reset_index())
display(statistics.style.hide(axis="index"))

Wu-Palmer similarity clearly separates the two relations in this table. Synonym pairs have a mean similarity of approximately 0.997 for nouns and 0.9999 for verbs, with minimum values of 0.73 and 0.67, respectively. Antonym pairs have much lower and more widely spread values. Noun antonyms have a mean similarity of 0.75 (range: 0.125–0.933), while verb antonyms have a mean of 0.37 (range: 0.111–0.857).

Synonym similarity is close to 1 by construction. WordNet synonyms are lemmas belonging to the same synset, meaning that we compare a synset with itself. The Wu-Palmer similarity of a synset with itself is 1. Values slightly below 1 are most likely edge cases in NLTK's implementation. Therefore, the synonym row mostly demonstrates that the measure behaves as expected rather than revealing anything new about synonymy. The antonym row is more informative.

Noun antonyms (mean: 0.75) tend to be contrasting members of the same category, such as man/woman or hot/cold concepts, and are relatively close to each other in the hierarchy. Verb antonyms have a much lower mean similarity (0.37). One possible explanation is that the verb hierarchy is relatively shallow and divided into many separate trees, meaning that opposites often end up far apart. However, this is something that should be investigated further rather than something directly demonstrated by our results.

Note that the code uses Wu-Palmer similarity (wup) and that there are approximately 100 times more synonym pairs than antonym pairs (107,469 vs. 1,076 for nouns and 24,577 vs. 548 for verbs).

 ## 2. Generate pair similarity distribution histograms

In [ ]:
BIN_COUNT = 10

for relation in RELATIONS:
    sims = DATAFRAME.loc[DATAFRAME["relation"] == relation, "similarity"].dropna()

    min_sim = sims.min()
    max_sim = sims.max()

    bins = np.linspace(min_sim, max_sim, BIN_COUNT + 1)

    plt.figure(figsize=(8,5))
    plt.hist(sims, bins=bins, edgecolor="black", rwidth=0.8, color=COLORS[relation, "n"])
    plt.xlabel("Pair similarity")
    plt.ylabel("Number of pairs")
    plt.title(f"{relation} similarity distribution")
    plt.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
    print()

The synonym histogram is extremely concentrated. The range extends from approximately 0.67 to 1.0, but nearly all ~130,000 pairs fall into the last bin, above 0.95, making the other bins barely visible. This is consistent with the near-1 mean reported in Section 1 and confirms that synonym similarity is almost degenerate.

The antonym histogram is spread across a much wider range (approximately 0.11 to 0.93) and appears to be bimodal. There is a smaller group around 0.2 to 0.35 and a larger group concentrated toward 0.8 to 0.93, with the highest bin containing the most pairs. This is consistent with the statistics in Section 1, where noun antonyms have a mean similarity of 0.75 and verb antonyms 0.37. Therefore, the two modes may correspond to the two parts of speech. Since the histograms combine nouns and verbs, plotting them separately by part of speech would help confirm this.

The two distributions overlap only at the upper end of the antonym range. Many antonym pairs, particularly those with higher similarity scores (mostly nouns), still reach values between 0.8 and 0.93 and could therefore be mistaken for synonyms when using a similarity threshold.

 ## 3. Difference between semantic similarity can be used distinguish between synonymy and antonymy

From the histograms above we can clearly see that there is huge difference in sematic similarity distribution between antonymy and synonymy.
In order to confirm this difference we can use t-test or more precisely **Independent Samples t-Test** to test if there is significant difference between two groups,
in your case sematic similarities values between antonyms and synonyms. Now, <br>

**Null Hypothesis**: *Both groups have identical sematic similarities and any observed difference is just noise.* <br>
**Alternative Hypothesis**: *The mean semantic similarities differ and difference is systematic.* <br>

using *Welch's t-test* formula, <br>
<br>
$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}} \qquad [1]
$$

In [ ]:
synonyms = DATAFRAME.loc[DATAFRAME["relation"] == "synonym", "similarity"].dropna()
antonyms = DATAFRAME.loc[DATAFRAME["relation"] == "antonym", "similarity"].dropna()

t, p = ttest_ind(synonyms, antonyms, equal_var=False)
print(f"t: {t:.3f}\np: {p}")

Now with result of **p-value** < 0.05 and high **t-value**, we can say results is statistically significant, meaning that we **reject** the Null Hypothesis and this provides support to claim that there is differences sematic similarities between synonyms and antonyms which we already distinguished from the histograms.

## 4. Transitivity relationships in synonyms and antonyms

In [ ]:
def build_relations_tables(df_pos):
    synonyms = defaultdict(set)
    antonyms = defaultdict(set)

    for row in df_pos.itertuples(index=False):
        table = synonyms if row.relation == "synonym" else antonyms
        table[row.word1].add(row.word2)
        table[row.word2].add(row.word1)

    return synonyms, antonyms


def get_transitivity_relationships(relations, synonyms):
    count = 0
    success = 0

    for y, neighbor in relations.items():
        for x, z in combinations(neighbor, 2):
            count += 1
            if z in synonyms.get(x, set()):
                success += 1

    return count, success


results = []

for pos in POS_TAGS:
    df_pos = DATAFRAME[DATAFRAME["pos"] == pos]

    synonyms, antonyms = build_relations_tables(df_pos)

    for relation, relations in [("synonym", synonyms), ("antonym", antonyms)]:
        total, successful = get_transitivity_relationships(relations, synonyms)

        results.append({
            'pos': pos,
            'relation': relation,
            'total': total,
            'successful': successful,
            'percentage': (successful / total) * 100 if total > 0 else 0
        })

df_transitivity = pd.DataFrame(results)
display(df_transitivity.groupby(["pos", "relation"]))

pivot = df_transitivity.pivot(
    index="relation",
    columns="pos",
    values="percentage"
)

ax = pivot.plot.bar(figsize=(8,5), edgecolor="black", rot=0)
for container, pos in zip(ax.containers, pivot.columns):
    for bar, relation in zip(container, pivot.index):
        bar.set_facecolor(COLORS[(relation, pos)])

ax.set_title("Transitivity of Synonymy and Antonymy")
ax.set_xlabel("Relationship")
ax.set_ylabel("Successful transitivity relationships (%)")
ax.set_ylim(0, 100)
ax.legend(["Nouns", "Verbs"])
legend_handles = [
    (
        Patch(facecolor=COLORS[('synonym', 'n')], edgecolor="white"),
        Patch(facecolor=COLORS[('antonym', 'n')], edgecolor="white")
    ),
    (
        Patch(facecolor=COLORS[('synonym', 'v')], edgecolor="white"),
        Patch(facecolor=COLORS[('antonym', 'v')], edgecolor="white")
    )
]

ax.legend(
    legend_handles,
    ["Nouns", "Verbs"],
    handler_map={tuple: HandlerTuple(ndivide=None)}
)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Transitivity does not hold reliably in WordNet when working at the word level. Only 70.2% of noun synonym triples (X~Y, Y~Z, with X and Z also checked) and 47.2% of verb triples satisfy X~Z.

Within a single synset, synonymy is transitive by definition, since a synset is a set of mutually interchangeable lemmas. Therefore, the failures almost certainly result from polysemy. Our tables are keyed by words rather than senses, meaning that X~Y can come from one sense of Y and Y~Z from another, while X and Z may have nothing to do with each other. Verbs are more polysemous than nouns in WordNet, which is a likely explanation for their much lower rate (47% vs. 70%). Highly polysemous hub words also generate a large number of triples, meaning that they dominate the overall counts (367,083 noun triples and 233,414 verb triples). Repeating the test at the sense (synset) level should produce values close to 100%, and this would be a useful way to verify the explanation.

For antonyms, the rule "the antonym of an antonym is a synonym" holds in only 29.9% of noun cases (20 of 67) and 23.1% of verb cases (21 of 91). This is not a reasonable general rule, because two words that share an antonym often have similar meanings without necessarily being listed as synonyms. The antonym samples are small, so these percentages should be interpreted with caution. Overall, neither relation is strongly transitive in WordNet, and antonymy is considerably less so.

## 5. Hypernymy in synonyms

In [ ]:
def has_common_hypernym(word1, word2, pos):
    for s1 in wn.synsets(word1, pos=pos):
        for s2 in wn.synsets(word2, pos=pos):
            if set(s1.hypernyms()) & set(s2.hypernyms()):
                return True
    return False


results = []

for pos in POS_TAGS:
    df_synonyms = DATAFRAME[
        (DATAFRAME["pos"] == pos) &
        (DATAFRAME["relation"] == "synonym")
    ]

    df_synonyms = df_synonyms.drop_duplicates(subset=["word1", "word2"])
    total = len(df_synonyms)

    hypernym_count = sum(
        has_common_hypernym(row.word1, row.word2, pos) for row in df_synonyms.itertuples(index=False)
    )

    results.append({
        'pos': "Noun" if pos == wn.NOUN else "Verb",
        'total': total,
        'shared hypernym count': hypernym_count,
        'percentage': (hypernym_count / total) * 100 if total > 0 else 0
    })

df_hypernyms_syn = pd.DataFrame(results)
display(df_hypernyms_syn.style.hide(axis="index"))

ax = df_hypernyms_syn.plot.bar(
    x="pos",
    y="percentage",
    figsize=(7, 4),
    legend=False,
    edgecolor="black",
    rot=0
)
for bar, pos in zip(ax.patches, df_hypernyms_syn["pos"]):
    bar.set_facecolor(COLORS[("synonym", wn.NOUN if pos == "Noun" else wn.VERB)])

ax.set_title("Synonym pairs sharing a hypernym")
ax.set_xlabel("Part of speech")
ax.set_ylabel("Pairs with shared hypernym (%)")
ax.set_ylim(0, 100)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

A total of 88.5% of noun synonym pairs and 94.0% of verb synonym pairs have at least one sense combination with a common direct hypernym. Therefore, the intuition that synonyms share a parent category is largely supported. After removing duplicate word pairs, the totals are 105,032 noun pairs and 23,020 verb pairs.

This result should be interpreted with two caveats. First, synonym pairs come from the same synset, so they trivially share hypernyms whenever that synset has any. The failing cases are therefore unlikely to be pairs that actually have different parents. Instead, they are more likely to come from synsets without a `hypernyms()` entry: in the case of nouns, probably instance synsets (e.g., named entities), which use `instance_hypernyms()` instead, and in the case of verbs, the many top-level verb synsets. We have not verified this, but checking `instance_hypernyms()` and counting synsets without hypernyms would help confirm it. Second, the test is existential across all senses of both words and considers only the direct parent. It is lenient in the first respect but strict in the second, meaning that the resulting percentages should not be considered highly precise.

## 6. Hypernymy in antonyms

In [ ]:
results = []

for pos in POS_TAGS:
    df_antonyms = DATAFRAME[
        (DATAFRAME["pos"] == pos) &
        (DATAFRAME["relation"] == "antonym")
    ]

    df_antonyms = df_antonyms.drop_duplicates(subset=["word1", "word2"])
    total = len(df_antonyms)

    hypernym_count = sum(
        has_common_hypernym(row.word1, row.word2, pos) for row in df_antonyms.itertuples(index=False)
    )

    results.append({
        'pos': "Noun" if pos == wn.NOUN else "Verb",
        'total': total,
        'shared hypernym count': hypernym_count,
        'percentage': (hypernym_count / total) * 100 if total > 0 else 0
    })

df_hypernyms_ant = pd.DataFrame(results)
display(df_hypernyms_ant.style.hide(axis="index"))

ax = df_hypernyms_ant.plot.bar(
    x="pos",
    y="percentage",
    figsize=(7, 4),
    legend=False,
    edgecolor="black",
    rot=0
)
for bar, pos in zip(ax.patches, df_hypernyms_ant["pos"]):
    bar.set_facecolor(COLORS[("antonym", wn.NOUN if pos == "Noun" else wn.VERB)])

ax.set_title("Antonym pairs sharing a hypernym")
ax.set_xlabel("Part of speech")
ax.set_ylabel("Pairs with shared hypernym (%)")
ax.set_ylim(0, 100)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Antonyms share a direct hypernym much less frequently than synonyms: 60.2% of noun pairs (597 of 992) and 34.5% of verb pairs (164 of 476). The difference compared to synonyms is approximately 28 percentage points for nouns (88.5% vs. 60.2%) and 60 percentage points for verbs (94.0% vs. 34.5%).

For nouns, a majority of antonym pairs share a parent, which supports the view that antonyms are contrasting members of the same category (opposite ends of the same dimension). This is consistent with their high Wu-Palmer similarity reported in Section 1 (mean: 0.75). For verbs, only about one-third share a parent, which aligns with their lower similarity (mean: 0.37). Verb opposites such as rise/fall or buy/sell are often placed in different parts of a relatively flat and fragmented hierarchy. As in Section 5, the test considers any sense combination of the two words rather than the specific antonym synsets. This probably overestimates the proportion of pairs sharing a hypernym, while the restriction to direct parents may underestimate it.

Taken together with the results from Section 1, the shared-hypernym test distinguishes synonyms from antonyms more clearly for verbs than for nouns. For nouns, antonyms remain relatively close to synonyms in terms of both similarity and hierarchical structure.

## 7. Word2Vec minimum and maximum distance

In [ ]:
word2vec = api.load("word2vec-google-news-300")

all_words = pd.unique(DATAFRAME[["word1", "word2"]].to_numpy().ravel())

def find_matching_w2v_word(word):
    candidates = [
        word,
        word.lower(),
        word.replace("_", " "),
        word.lower().replace("_", " "),
        word.title()
    ]

    for candidate in candidates:
        if candidate in word2vec:
            return candidate
    return None


word_mapping = {
    word: find_matching_w2v_word(word) for word in all_words
}

valid_w2v_words = {
    word: mapped for word, mapped in word_mapping.items() if mapped is not None
}

print("Total unique words:", len(all_words))
print("Matching words found in Word2Vec:", len(valid_w2v_words))
print(f"Coverage: {len(valid_w2v_words) / len(all_words) * 100:.2f}%")

embedding_words = list(dict.fromkeys(valid_w2v_words.values()))

X = np.array(
    [word2vec[word] for word in embedding_words],
    dtype=np.float32
)

# reduce dimensions from original X from (N,300) to (N,50)
PCA_reduced_X = PCA(
    n_components=min(50, X.shape[0], X.shape[1]),
    random_state=2026
).fit_transform(X)

# project PCA reduced X from (N,50) into 2D
X_2d_projection = TSNE(
    n_components=2,
    perplexity=min(30, X.shape[0] - 1),
    random_state=2026,
    init="pca",
    learning_rate="auto"
).fit_transform(PCA_reduced_X)

word_coordinates_2d = dict(zip(embedding_words, X_2d_projection))

def distance_2d(w1, w2):
    if w1 not in word_coordinates_2d or w2 not in word_coordinates_2d:
        return np.nan
    return np.linalg.norm(word_coordinates_2d[w1] - word_coordinates_2d[w2])


DATAFRAME["distance_2d"] = [distance_2d(w1, w2) for w1, w2 in zip(DATAFRAME["word1"], DATAFRAME["word2"])]


df_unique_pairs = (
    DATAFRAME.dropna(subset=["distance_2d"])
    .drop_duplicates(subset=["pos", "relation", "word1", "word2"]).copy()
)

minmax_2d_dists = []

for pos in POS_TAGS:
    for relation in RELATIONS:
        group = df_unique_pairs[
            (df_unique_pairs["relation"] == relation) & (df_unique_pairs["pos"] == pos)
        ]
        min_row = group.loc[[group["distance_2d"].idxmin()]].iloc[0]
        max_row = group.loc[[group["distance_2d"].idxmax()]].iloc[0]
        minmax_2d_dists.append({
            'pos': pos,
            'relation': relation,
            "extreme": "min",
            'word1': min_row["word1"],
            'word2': min_row["word2"],
            'distance_2d': min_row["distance_2d"]
        })
        minmax_2d_dists.append({
            'pos': pos,
            'relation': relation,
            "extreme": "max",
            'word1': max_row["word1"],
            'word2': max_row["word2"],
            'distance_2d': max_row["distance_2d"]
        })

df_minmax_2d_dists = pd.DataFrame(minmax_2d_dists)


# Everything below this is just for displaying the results
# ---
def plot_min_max_for_group(df_group, coords, pos, relation):
    min_row = df_group.loc[df_group["distance_2d"].idxmin()]
    max_row = df_group.loc[df_group["distance_2d"].idxmax()]
    isMax = False
    plt.figure(figsize=(7, 7))

    for row in [min_row, max_row]:
        w1 = row["word1"]
        w2 = row["word2"]
        x1, y1 = coords[w1]
        x2, y2 = coords[w2]
        xytexts = [(10, -10), (-10, 5)]
        ha = ["left", "right"]

        plt.scatter([x1, x2], [y1, y2], s=80)
        plt.plot([x1, x2], [y1, y2], linewidth=1.5)

        if isMax and row["pos"] == wn.NOUN:
            xytexts = [(-10, -20), (10, 15)]

        if isMax and row["pos"] == wn.VERB:
            xytexts = [(-20, -15), (5, 8)]
            ha = ["left", "left"]

        plt.annotate(
            w1, (x1, y1),
            xytext=xytexts[0],
            textcoords="offset points",
            ha=ha[0],
            fontsize=10
        )
        plt.annotate(
            w2, (x2, y2),
            xytext=xytexts[1],
            textcoords="offset points",
            ha=ha[1],
            fontsize=10
        )
        isMax = True

    title_pos = "Noun" if pos == "n" else "Verb"
    plt.title(f"{relation.capitalize()} pairs [{title_pos}]\nMinimum and maximum (Word2Vec) 2D distances ")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim(-100, 100)
    plt.ylim(-100, 100)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print()

for pos in ["n", "v"]:
    for relation in ["synonym", "antonym"]:
        group = df_minmax_2d_dists[
            (df_minmax_2d_dists["pos"] == pos) &
            (df_minmax_2d_dists["relation"] == relation)
        ]
        display(group.style.hide(axis="index"))
        if not group.empty:
            plot_min_max_for_group(group, word_coordinates_2d, pos, relation)

We embedded WordNet words using pretrained Word2Vec vectors. We reduced the original 300 dimensions to 50 dimensions using PCA and then projected them into 2D using t-SNE. Finally, we measured the Euclidean distance between the words in each pair.

Minimum distances: The closest pairs are almost at the same point, with distances ranging from approximately 0.001 to 0.006 for both relations. For synonyms, this is expected, since the closest pairs are either spelling variants or nearly identical expressions. Examples include chaperon/chaperone and klan/Ku_Klux_Klan. The surprising result is that the closest antonym pairs, leeward/windward and emigrate/immigrate, are just as close. Word2Vec is trained on word co-occurrence, and antonyms often appear in similar contexts. Consequently, the embeddings treat them as near-neighbours. This is a known limitation of distributional embeddings: they capture semantic relatedness rather than opposition.

Maximum distances: The furthest pairs are not clearly separated by relation either. The maximum distances for synonyms (167.2 and 162.2) are greater than those for antonyms (133.8 and 119.2). One possible explanation is that there are approximately 100 times more synonym pairs than antonym pairs, making extreme values more likely to occur among synonyms. The extreme pairs may also be artefacts rather than genuine examples of semantic distance.

Overall, the minimum and maximum distances do not distinguish synonymy from antonymy. Both relations contain pairs with near-zero distances as well as pairs that are far apart. The minimum and maximum values are heavily influenced by outliers and sample size, as the synonym set is considerably larger. Therefore, they reveal little about typical behaviour. A comparison of mean and median distances, or of the full distance distributions using a statistical test, would provide more informative results.

## 8. Synonymy and antonymy analysis outside WordNet Lexicon database

In [ ]:
response = requests.get(URL, timeout=15)
response.raise_for_status()


def parse_page(resp):
    text = resp.text

    # Split the dictionary into individual entries
    entries = text.split("=")

    synonym_pairs = set()
    antonym_pairs = set()

    for entry in entries:

        # Find the main word
        key_match = re.search(r"KEY:\s*(.+)", entry, re.IGNORECASE)

        if key_match is None:
            continue

        word1 = key_match.group(1).strip()

        # Remove notes such as [See MERCY] and part-of-speech marks
        word1 = re.sub(r"\[.*?\]", "", word1)
        word1 = re.sub(r"\\[va]\.", "", word1, flags=re.IGNORECASE)
        word1 = word1.strip(" .\\").lower()

        # Find synonym and antonym lists
        for label, pair_set in [
            ("SYN", synonym_pairs),
            ("ANT", antonym_pairs)
        ]:

            # Read text after the label until the next label or end
            match = re.search(
                rf"{label}:\s*(.*?)(?=\n\s*(?:SYN|ANT|KEY):|\Z)",
                entry,
                re.IGNORECASE | re.DOTALL
            )

            if match is None:
                continue

            words = re.findall(r"[A-Za-z]+", match.group(1).lower())

            for word2 in words:
                if word1 != word2:
                    pair_set.add(tuple(sorted((word1, word2))))

    return synonym_pairs, antonym_pairs


def word_similarity(word1, word2, pos):
    scores = [
        similarity(s1.name(), s2.name())
        for s1 in wn.synsets(word1, pos=pos)
        for s2 in wn.synsets(word2, pos=pos)
    ]

    # Filter none scores
    scores = [score for score in scores if score is not None]

    return max(scores) if scores else None

syn_pairs, ant_pairs = parse_page(response)

results = []
wn_pairs = set(zip(DATAFRAME["pos"],
                   DATAFRAME["relation"],
                   DATAFRAME["word1"],
                   DATAFRAME["word2"]))

for relation, pairs in [("synonym", syn_pairs), ("antonym", ant_pairs)]:
    for word1, word2 in pairs:
        for pos in POS_TAGS:

            # Both words must have this part of speech
            if not wn.synsets(word1, pos=pos):
                continue

            if not wn.synsets(word2, pos=pos):
                continue

            key = (pos, relation, word1, word2)

            # Keep only relationships missing from WordNet Lexicon dataframe
            if key in wn_pairs:
                continue

            score = word_similarity(word1, word2, pos)

            results.append({
                "relation": relation,
                "pos": pos,
                "word1": word1,
                "word2": word2,
                "similarity": score
            })
            wn_pairs.add(key)

df_gutenberg = pd.DataFrame(results)
print("Missing relationships:", len(df_gutenberg))
statistics = (df_gutenberg.groupby(["pos", "relation"])["similarity"]
              .agg(["count", "min", "max", "mean"]).reset_index())
display(statistics.style.hide(axis="index"))

After filtering, 14,258 noun synonym pairs, 113,733 noun antonym pairs, 11,607 verb synonym pairs, and 10,172 verb antonym pairs from Fallows' dictionary were missing from the WordNet pair list (which contains approximately 47,000 pairs in total). The antonym numbers stand out the most, since there are approximately ten times more missing antonym pairs than the total number of antonym pairs in WordNet. WordNet records antonymy only through direct, sense-specific lexical links, so a dictionary of words with opposite meanings will naturally contain many conceptual opposites that WordNet does not explicitly link. At the same time, part of this volume is almost certainly due to parsing noise. Therefore, the counts should not be interpreted as a clean measure of WordNet's coverage gap.

All four groups have low mean similarity scores: 0.15 for noun antonyms, 0.23 for noun synonyms, 0.21 for verb antonyms, and 0.29 for verb synonyms. The synonym means are far below the WordNet synonym ranges reported in Section 1 (minimum values of 0.73 for nouns and 0.67 for verbs, with means close to 1). This indicates that the dictionary synonym pairs do not exhibit the same similarity levels as WordNet synonym pairs. WordNet synonyms are lemmas belonging to the same synset, so their similarity is approximately 1 by construction, whereas Fallows' synonyms are mostly near-synonyms or loosely related words that belong to different synsets.

The antonym minimum values (0.045 and 0.053) are below the WordNet antonym minimum values (0.125 and 0.111, respectively, as reported in Section 1). The maximum values are even more interesting, as they are all equal to 1.0 across all four groups. These values exceed the WordNet antonym maximum values (0.933 for nouns and 0.857 for verbs). A score of 1.0 means that at least one sense of the first word and one sense of the second word belong to the same synset. For antonyms, there are two likely explanations. First, polysemy: since `word_similarity` takes the maximum across all sense pairs, two words can be antonyms in one sense but synonyms in another. Second, parsing errors may introduce incorrect word pairs.

For both parts of speech, antonyms have lower mean similarity scores than synonyms (nouns: 0.15 vs. 0.23; verbs: 0.21 vs. 0.29). This follows the same pattern as in WordNet, although the difference is relatively small (approximately 0.07). In WordNet, the difference between noun synonyms and antonyms is considerably larger (approximately 0.24 using Wu-Palmer similarity). The noun/verb ordering for antonyms is also reversed compared to WordNet, where noun antonyms (0.75) had considerably higher similarity than verb antonyms (0.37).

Overall, the Gutenberg-only pairs are less closely related in WordNet than WordNet's own pairs. This is partly due to genuine differences, such as the looser definitions of synonymy and antonymy used in the dictionary, and partly due to artefacts in the experimental setup. These conclusions should be verified by rerunning the analysis using Wu-Palmer similarity and cleaning the parsed data.

## 9. State of the art technology in synonymy and antonymy research

- Synonymy and antonymy are fundamental semantic relationships in natural language processing (NLP). Synonymy refers to words with similar meanings, whereas antonymy describes words with opposite meanings. Several studies have investigated how these relationships can be identified and represented using lexical databases and computational models.

- Miller et al. (1990) introduced WordNet, a lexical database that organizes words into synonym sets called synsets. WordNet also represents semantic relationships such as antonymy, hypernymy, and hyponymy. It provides an important foundation for computational semantic analysis. However, WordNet does not include every possible synonym or antonym, and relationships may depend on the specific meaning of a word. Therefore, its classifications are not always comprehensive.

- Maziarz, Piasecki, and Szpakowicz (2013) examined the theoretical challenges involved in constructing WordNet. Their research highlights the differences between synonymy and antonymy. While synonymy is represented through groups of words sharing similar meanings, antonymy generally connects individual lexical items. This distinction is important when examining whether these relationships follow mathematical properties such as transitivity.
- Mohammad et al. (2013) further investigated lexical contrast and demonstrated the importance of considering opposing semantic dimensions when identifying antonyms. This suggests that semantic similarity alone is insufficient to distinguish synonyms from antonyms. For example, hot and cold are antonyms but still share a strong semantic relationship.

- Wu and Palmer (1994) introduced a semantic similarity measure based on hierarchical relationships between concepts. Their approach calculates similarity using the depth of concepts and their lowest common subsumer in a taxonomy. This is particularly relevant to WordNet, where words can share hypernyms despite having opposite meanings. Therefore, high semantic similarity does not necessarily indicate synonymy.

- Finally, Mikolov et al. (2013) introduced Word2Vec, a method for representing words as numerical vectors based on their surrounding linguistic contexts. Word2Vec makes it possible to compare synonyms and antonyms using vector distances. However, because antonyms often appear in similar contexts, their vector representations may also be close to each other.

## Use of AI
OpenAI model GPT-6 Astra was used help for code debugging and getting 2D projection of Word2Vec space as well as getting correct formatting for graphs. Also used for checking the grammar of analysis parts.

## Sources
[1] https://numiqo.com/tutorial/t-test

Miller, G. A., et al. (1990). Introduction to WordNet: An On-Line Lexical Database. International Journal of Lexicography, 3(4), 235–244.​

Maziarz, M., Piasecki, M., & Szpakowicz, S. (2013). The Chicken-and-Egg Problem in WordNet Design: Synonymy, Synsets and Constitutive Relations. Language Resources and Evaluation, 47, 769–796.​

Mohammad, S. M., et al. (2013). Computing Lexical Contrast. Computational Linguistics, 39(3), 555–590.​

Wu, Z., & Palmer, M. (1994). Verb Semantics and Lexical Selection. ACL 1994.​

Mikolov, T., et al. (2013). Efficient Estimation of Word Representations in Vector Space. ICLR 2013.​

​